# Train DeepONet on Fluent cell-center subdomains

This notebook uses the non-grid dataset. The trunk queries are original Fluent cell centers inside each subdomain, not a 256×256 raster grid. Branch sensors are exterior BC points plus interpolated interface points.

In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())
print("NCCL:", torch.cuda.nccl.version())
print("GPU count:", torch.cuda.device_count())

PyTorch: 2.4.1
CUDA: 12.4
cuDNN: 90100
NCCL: (2, 20, 5)
GPU count: 8


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import builtins
import json
import sys
import logging
import subprocess
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from datetime import datetime

MODULE_DIR = Path.cwd()
if not (MODULE_DIR / "fluent_deeponet.py").exists():
    MODULE_DIR = MODULE_DIR / "DeepONet"
ROOT_DIR = MODULE_DIR.parent

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from deeponet_fluent_dataset import (
    build_fluent_deeponet_dataset,
    DeepONetCellDataset,
    deeponet_cell_collate_fn,
)

from fluent_deeponet import (
    FeatureNormalizer,
    DeepONet,
    train_deeponet_one_epoch,
    evaluate_deeponet,
)

from plot import (
    plot_prediction_imshow_from_points,
    collect_predictions_for_data,
)

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda:1


In [2]:
comment = "physics informed NO, 16k training samples, 1k epochs"

In [3]:
def save_git_snapshot(run_dir):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    commands = {
        "git_commit.txt": ["git", "rev-parse", "HEAD"],
        "git_status.txt": ["git", "status"],
        "git_diff.patch": ["git", "diff"],
        "git_diff_cached.patch": ["git", "diff", "--cached"],
    }

    for filename, command in commands.items():
        with open(run_dir / filename, "w", encoding="utf-8") as f:
            subprocess.run(command, stdout=f, stderr=subprocess.STDOUT)

today = datetime.now().strftime("%m%d%y")
output_root = MODULE_DIR / "results"
output_root.mkdir(parents=True, exist_ok=True)
run_no = len([s for s in output_root.iterdir() if today in s.name]) + 1
output_str = f"{today}_{run_no}"
output_dir = output_root / output_str
output_dir.mkdir(parents=True, exist_ok=True)
save_git_snapshot(output_dir / "code_snapshot")
logger = logging.getLogger("pino")
logging.basicConfig(filename=output_dir / 'training.log', 
                    level=logging.INFO, 
                    force=True,
                    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S')

orig_print = builtins.print
def print(*args, sep=' ', end='\n', file=None, flush=False):
    orig_print(*args, sep=sep, end=end, file=file, flush=flush)
    msg = sep.join(map(str, args))
    logger.info(msg)
    
print("Training comment")
print(comment)

Training comment
physics informed NO, 16k training samples, 1k epochs


## Case file paths

In [4]:
ROOT_DIR = Path("/home/hantianl/Documents/PIDIF/")
DATASET_NAME = "channel_water"

def case_paths(ch):
    return {
        "design": ROOT_DIR / f"2d_geometry_specs/{DATASET_NAME}/{ch}.json",
        "mesh": ROOT_DIR / f"runs_2d/{DATASET_NAME}/{ch}/{ch}.msh.h5",
        "dat": ROOT_DIR / f"runs_2d/{DATASET_NAME}/{ch}/case2d.dat.h5",
    }

# Keep only cases whose design/mesh/dat files all exist (completed runs).
available_cases = [
    ch.name for ch in (ROOT_DIR / "runs_2d" / DATASET_NAME).iterdir()
    if all(case_paths(ch.name)[k].exists() for k in ("design", "mesh", "dat"))
]
case_files = {ch: case_paths(ch) for ch in available_cases}

print(f"Available cases ({len(available_cases)}):")
print(sorted([int(ch.split("_")[-1]) for ch in available_cases]))

Available cases (200):
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199]


# Training config

In [5]:
# 80/10/10 train/val/test split over the available cases (split by case id).
SPLIT_SEED = 0
_split_rng = np.random.default_rng(SPLIT_SEED)
_shuffled_cases = list(available_cases)
_split_rng.shuffle(_shuffled_cases)

n_total = len(_shuffled_cases)
n_train = int(round(0.80 * n_total))
n_valid = int(round(0.10 * n_total))
TRAIN_CASES = sorted(_shuffled_cases[:n_train], key=lambda x: int(x.split("_")[-1]))
VALID_CASES = sorted(_shuffled_cases[n_train:n_train + n_valid], key=lambda x: int(x.split("_")[-1]))
TEST_CASES = sorted(_shuffled_cases[n_train + n_valid:], key=lambda x: int(x.split("_")[-1]))

print(f"Train cases ({len(TRAIN_CASES)}):", TRAIN_CASES)
print(f"Valid cases ({len(VALID_CASES)}):", VALID_CASES)
print(f"Test  cases ({len(TEST_CASES)}):", TEST_CASES)

# Interfaces at fixed locations; n_sub adapts to each channel's aspect ratio.
N_SUBDOMAINS = "adaptive"
N_INTERFACE_POINTS = 256
N_WALL_POINTS = 256
INTERFACE_PLACEMENT = "random"
INTERFACE_JITTER = 0.0
MIN_SUBDOMAIN_WIDTH = 0.01
N_REALIZATIONS_TRAIN = 10
HORIZONTAL_INTERFACE = False
HORIZONTAL_INTERFACE_JITTER = 0.01

N_QUERY_TRAIN = None
N_QUERY_VALID = None
BATCH_SIZE = 50

# Isothermal runs have no temperature field: train on pressure/u/v only.
FIELD_MAP = {"pressure": "SV_P", "u": "SV_U", "v": "SV_V"}

LAMBDA_BC = 1.0
LAMBDA_PDE = 1.0
N_PDE_POINTS = 10000  # Random interior/cell-center collocation points per sample.
PDE_RESIDUAL_WEIGHTS = {
    "continuity": 1.0,
    "x_momentum": 1.0,
    "y_momentum": 1.0,
}
WATER_DENSITY = 998.2  # kg/m^3
WATER_KINEMATIC_VISCOSITY = 1.003e-3 / WATER_DENSITY  # m^2/s

bc_kwargs = {
    "inlet_v": 0.0,
    "outlet_p": 0.0,
    "wall_u": 0.0,
    "wall_v": 0.0,
}

Train cases (160): ['channel_00', 'channel_01', 'channel_02', 'channel_03', 'channel_04', 'channel_05', 'channel_06', 'channel_07', 'channel_09', 'channel_10', 'channel_11', 'channel_12', 'channel_13', 'channel_14', 'channel_15', 'channel_16', 'channel_17', 'channel_19', 'channel_20', 'channel_21', 'channel_23', 'channel_24', 'channel_25', 'channel_26', 'channel_27', 'channel_29', 'channel_30', 'channel_32', 'channel_36', 'channel_38', 'channel_39', 'channel_40', 'channel_41', 'channel_42', 'channel_43', 'channel_44', 'channel_46', 'channel_47', 'channel_49', 'channel_52', 'channel_54', 'channel_55', 'channel_56', 'channel_57', 'channel_58', 'channel_59', 'channel_60', 'channel_61', 'channel_63', 'channel_64', 'channel_65', 'channel_66', 'channel_67', 'channel_68', 'channel_69', 'channel_70', 'channel_72', 'channel_73', 'channel_74', 'channel_76', 'channel_77', 'channel_78', 'channel_79', 'channel_82', 'channel_83', 'channel_84', 'channel_85', 'channel_86', 'channel_88', 'channel_91', 

## Build non-grid training/validation dataset

In [ ]:
train_data = build_fluent_deeponet_dataset(
    case_files=case_files,
    case_ids=TRAIN_CASES,
    n_subdomains=N_SUBDOMAINS,
    n_interface_points=N_INTERFACE_POINTS,
    n_boundary_points=N_WALL_POINTS,
    n_realizations=N_REALIZATIONS_TRAIN,
    interface_placement=INTERFACE_PLACEMENT,
    interface_jitter=INTERFACE_JITTER,
    min_subdomain_width=MIN_SUBDOMAIN_WIDTH,
    horizontal_interface=HORIZONTAL_INTERFACE,
    insert_sharp_control_point_interfaces=False,
    field_map=FIELD_MAP,
    bc_kwargs=bc_kwargs,
    keep_raw_case_data=False,
    rng=rng,
)

valid_data = build_fluent_deeponet_dataset(
    case_files=case_files,
    case_ids=VALID_CASES,
    n_subdomains=N_SUBDOMAINS,
    n_interface_points=N_INTERFACE_POINTS,
    n_boundary_points=N_WALL_POINTS,
    interface_placement=INTERFACE_PLACEMENT,
    interface_jitter=0,
    min_subdomain_width=MIN_SUBDOMAIN_WIDTH,
    horizontal_interface=HORIZONTAL_INTERFACE,
    insert_sharp_control_point_interfaces=False,
    field_map=FIELD_MAP,
    bc_kwargs=bc_kwargs,
    keep_raw_case_data=False,
    rng=rng,
)

samples = train_data["samples"]
metadata = train_data["metadata"]
branch_channel_names = list(train_data["branch_channel_names"])
trunk_channel_names = list(train_data["trunk_channel_names"])
output_channel_names = list(train_data["output_channel_names"])

ars = np.asarray([int(round(m["aspect_ratio"])) for m in metadata], dtype=np.int64)
subdomain_ids = np.asarray([int(m["subdomain_id"]) for m in metadata], dtype=np.int64)
local_aspect_ratios = np.asarray([float(m["local_aspect_ratio"]) for m in metadata], dtype=np.float32)
n_cells = np.asarray([int(m["n_cells"]) for m in metadata], dtype=np.int64)

print("Train samples:", len(train_data["samples"]))
print("Valid samples:", len(valid_data["samples"]))

print("Train branch channels:", train_data["branch_channel_names"])
print("Train trunk channels:", train_data["trunk_channel_names"])
print("Output channels:", train_data["output_channel_names"])

print("Example train sample:")
print("branch:", train_data["samples"][0]["branch"].shape)
print("query:", train_data["samples"][0]["query"].shape)
print("target:", train_data["samples"][0]["target"].shape)

Processing case channel_00 realization=0 (n_subdomains=10)
Processing case channel_00 realization=1 (n_subdomains=10)
Processing case channel_00 realization=2 (n_subdomains=10)
Processing case channel_00 realization=3 (n_subdomains=10)
Processing case channel_00 realization=4 (n_subdomains=10)
Processing case channel_00 realization=5 (n_subdomains=10)
Processing case channel_00 realization=6 (n_subdomains=10)
Processing case channel_00 realization=7 (n_subdomains=10)
Processing case channel_00 realization=8 (n_subdomains=10)
Processing case channel_00 realization=9 (n_subdomains=10)
Processing case channel_01 realization=0 (n_subdomains=10)
Processing case channel_01 realization=1 (n_subdomains=10)
Processing case channel_01 realization=2 (n_subdomains=10)
Processing case channel_01 realization=3 (n_subdomains=10)
Processing case channel_01 realization=4 (n_subdomains=10)
Processing case channel_01 realization=5 (n_subdomains=10)
Processing case channel_01 realization=6 (n_subdomains=1

In [ ]:
train_metadata = train_data["metadata"]
valid_metadata = valid_data["metadata"]

train_subdomain_ids = np.asarray(
    [m["subdomain_id"] for m in train_metadata],
    dtype=np.int64,
)

valid_subdomain_ids = np.asarray(
    [m["subdomain_id"] for m in valid_metadata],
    dtype=np.int64,
)

train_case_ids = sorted({int(m["case_id"].split("_")[-1]) for m in train_metadata})
valid_case_ids = sorted({int(m["case_id"].split("_")[-1]) for m in valid_metadata})

print("Train cases:", train_case_ids)
print("Valid cases:", valid_case_ids)

Train cases: [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 23, 24, 25, 26, 27, 29, 30, 32, 36, 38, 39, 40, 41, 42, 43, 44, 46, 47, 49, 52, 54, 55, 56, 57, 58, 59, 60, 61, 63, 64, 65, 66, 67, 68, 69, 70, 72, 73, 74, 76, 77, 78, 79, 82, 83, 84, 85, 86, 88, 91, 92, 93, 94, 95, 96, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 112, 113, 114, 115, 116, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 134, 135, 136, 137, 138, 139, 140, 143, 144, 145, 146, 147, 148, 149, 150, 152, 153, 155, 156, 157, 158, 159, 160, 161, 162, 165, 166, 167, 168, 169, 170, 173, 174, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 192, 194, 195, 196, 197, 198]
Valid cases: [8, 34, 35, 37, 48, 50, 53, 62, 75, 80, 87, 141, 142, 154, 163, 164, 171, 175, 176, 191]


In [ ]:
def concatenate_targets(data):
    return np.concatenate(
        [sample["target"] for sample in data["samples"]],
        axis=0,
    ).astype(np.float32)


train_targets_all = concatenate_targets(train_data)

y_normalizer = FeatureNormalizer(train_targets_all)

print("Y mean:", y_normalizer.mean)
print("Y std:", y_normalizer.std)

train_local_aspects = np.asarray(
    [m["local_aspect_ratio"] for m in train_data["metadata"]],
    dtype=np.float32,
)

local_aspect_mean = float(train_local_aspects.mean())
local_aspect_std = float(train_local_aspects.std() + 1.0e-6)

print("local_aspect_mean:", local_aspect_mean)
print("local_aspect_std:", local_aspect_std)

# Per-sample physical scales for the nondimensional steady NS residual.
# Dataset metadata lengths are in mm; the PDE helper expects metres.
_case_velocity = {}
for case_id in TRAIN_CASES:
    with open(case_files[case_id]["design"], "r", encoding="utf-8") as f:
        _case_velocity[case_id] = float(json.load(f)["metadata"]["Uin_mps"])

pde_physics = {
    "density": WATER_DENSITY,
    "kinematic_viscosity": WATER_KINEMATIC_VISCOSITY,
    "x_length": np.asarray(
        [(m["x_right_mm"] - m["x_left_mm"]) * 1.0e-3 for m in train_data["metadata"]],
        dtype=np.float32,
    ),
    "y_length": np.asarray(
        [(m["y_top_mm"] - m["y_bottom_mm"]) * 1.0e-3 for m in train_data["metadata"]],
        dtype=np.float32,
    ),
    "velocity_scale": np.asarray(
        [_case_velocity[m["case_id"]] for m in train_data["metadata"]],
        dtype=np.float32,
    ),
}

print("PDE Reynolds-number range:",
      float(np.min(pde_physics["velocity_scale"] * pde_physics["x_length"] / WATER_KINEMATIC_VISCOSITY)),
      float(np.max(pde_physics["velocity_scale"] * pde_physics["x_length"] / WATER_KINEMATIC_VISCOSITY)))

Y mean: tensor([ 6.4387e+02,  1.0012e+00, -7.9760e-06])
Y std: tensor([4.1250e+02, 4.2776e-01, 2.2414e-02])
local_aspect_mean: 1.0
local_aspect_std: 9.999999974752427e-07
PDE Reynolds-number range: 99.52143859863281 99.52143859863281


## DataLoaders

In [ ]:
train_ds = DeepONetCellDataset(
    samples=train_data["samples"],
    sample_indices=None,
    n_query_points=N_QUERY_TRAIN,
    random_query=True,
    target_y_normalizer=y_normalizer,
    local_aspect_mean=local_aspect_mean,
    local_aspect_std=local_aspect_std,
    branch_channel_names=train_data["branch_channel_names"],
)

valid_ds = DeepONetCellDataset(
    samples=valid_data["samples"],
    sample_indices=None,
    n_query_points=N_QUERY_VALID,
    random_query=False,
    target_y_normalizer=y_normalizer,
    local_aspect_mean=local_aspect_mean,
    local_aspect_std=local_aspect_std,
    branch_channel_names=valid_data["branch_channel_names"],
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    collate_fn=deeponet_cell_collate_fn,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    collate_fn=deeponet_cell_collate_fn,
)

branch_batch, query_batch, target_batch, query_batch_id, sample_idx_batch, branch_mask_batch = next(iter(train_loader))

print("branch batch:", branch_batch.shape)
print("query batch:", query_batch.shape)
print("target batch:", target_batch.shape)
print("query batch id:", query_batch_id.min(), query_batch_id.max())
print("sample indices:", sample_idx_batch)
print("branch mask batch:", branch_mask_batch.shape)


branch batch: torch.Size([50, 1024, 11])
query batch: torch.Size([1160255, 2])
target batch: torch.Size([1160255, 3])
query batch id: tensor(0) tensor(49)
sample indices: tensor([ 826,  346,  329,  816, 1153,  536,  785, 1472,   72,  271,  661,  111,
        1418, 1169, 1121, 1098,  883,  691,  279,  484, 1308,  931,  541,  617,
         419,  109, 1416,   16, 1335,  238, 1096, 1061, 1175,  604,  616, 1052,
         303, 1490,  217, 1143,  871, 1555,  928, 1537, 1559,  933,  856,  648,
          30, 1330])
branch mask batch: torch.Size([50, 1024])


## Model

In [ ]:
branch_input_dim = train_data["samples"][0]["branch"].shape[-1]
trunk_input_dim = train_data["samples"][0]["query"].shape[-1]
output_channels = train_data["samples"][0]["target"].shape[-1]

model = DeepONet(
    branch_input_dim=branch_input_dim,
    trunk_input_dim=trunk_input_dim,
    output_channels=output_channels,
    latent_dim=128,
    branch_point_hidden_dim=128,
    branch_point_depth=3,
    branch_global_hidden_dim=128,
    branch_global_depth=3,
    trunk_hidden_dim=128,
    trunk_depth=4,
    aggregation="mean",
    activation="gelu",
    layer_norm=False,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1.0e-3, weight_decay=1.0e-4)
EPOCHS = 1000
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(model.config())

{'branch_input_dim': 11, 'trunk_input_dim': 2, 'output_channels': 3, 'latent_dim': 128, 'branch_point_hidden_dim': 128, 'branch_point_depth': 3, 'branch_global_hidden_dim': 128, 'branch_global_depth': 3, 'trunk_hidden_dim': 128, 'trunk_depth': 4, 'aggregation': 'mean', 'activation': 'gelu', 'layer_norm': False}


## Train

In [ ]:
checkpoint_path = output_dir / "checkpoint.pt"

LOSS_TYPE = "mse"

best_valid_mse = float("inf")
history = []
y_normalizer = y_normalizer.to(DEVICE)

start = datetime.now()
for epoch in range(1, EPOCHS + 1):
    data_loss, bc_loss, pde_loss = train_deeponet_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        device=DEVICE,
        loss_type=LOSS_TYPE,
        lambda_bc=LAMBDA_BC,
        lambda_pde=LAMBDA_PDE,
        branch_channel_names=branch_channel_names,
        output_channel_names=output_channel_names,
        y_normalizer=y_normalizer,
        pde_physics=pde_physics,
        pde_residual_weights=PDE_RESIDUAL_WEIGHTS,
        n_pde_points=N_PDE_POINTS,
    )
    scheduler.step()

    valid_metrics = evaluate_deeponet(
        model=model,
        loader=valid_loader,
        device=DEVICE,
        y_normalizer=y_normalizer,
    )
    valid_mse = valid_metrics["mse"]
    valid_rel = valid_metrics["relative_l2"]
    valid_channel_rel = valid_metrics["channel_relative_l2"]

    history.append({
        "epoch": epoch,
        "data_loss": data_loss,
        "bc_loss": bc_loss,
        "pde_loss": pde_loss,
        "valid_mse": valid_mse,
        "valid_relative_l2": valid_rel,
        "valid_channel_relative_l2": valid_channel_rel,
    })

    if valid_mse < best_valid_mse:
        best_valid_mse = valid_mse
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_config": model.config(),
                "y_normalizer": y_normalizer.state_dict(),
                "branch_channel_names": train_data["branch_channel_names"],
                "trunk_channel_names": train_data["trunk_channel_names"],
                "output_channel_names": train_data["output_channel_names"],
                "train_cases": TRAIN_CASES,
                "valid_cases": VALID_CASES,
                "test_cases": TEST_CASES,
                "n_subdomains": N_SUBDOMAINS,
                "field_map": FIELD_MAP,
                "n_interface_points": N_INTERFACE_POINTS,
                "n_wall_points": N_WALL_POINTS,
                "interface_placement": INTERFACE_PLACEMENT,
                "interface_jitter": INTERFACE_JITTER,
                "min_subdomain_width": MIN_SUBDOMAIN_WIDTH,
                "horizontal_interface": HORIZONTAL_INTERFACE,
                "local_aspect_mean": local_aspect_mean,
                "local_aspect_std": local_aspect_std,
                "bc_kwargs": bc_kwargs,
                "lambda_pde": LAMBDA_PDE,
                "n_pde_points": N_PDE_POINTS,
                "pde_residual_weights": PDE_RESIDUAL_WEIGHTS,
                "pde_physics": pde_physics,
            },
            checkpoint_path,
        )

    if epoch == 1 or epoch % 10 == 0:
        channel_msg = ", ".join(f"{name}={err:.3e}" for name, err in zip(output_channel_names, valid_channel_rel))
        print(
            f"Epoch {epoch:04d} | data loss={data_loss:.3e} | bc loss={bc_loss:.3e} | "
            f"PDE loss={pde_loss:.3e} | "
            f"valid MSE={valid_metrics['mse']:.3e} | valid rel L2={valid_rel:.3e} | {channel_msg} | "
            f"Time: {datetime.now() - start}"
        )

print("Best validation MSE:", best_valid_mse)
print("Saved:", checkpoint_path)

loss_history_path = output_dir / f"loss_history.npz"
np.savez(
    loss_history_path,
    epoch=np.asarray([h["epoch"] for h in history], dtype=np.int32),
    data_loss=np.asarray([h["data_loss"] for h in history], dtype=np.float32),
    bc_loss=np.asarray([h["bc_loss"] for h in history], dtype=np.float32),
    pde_loss=np.asarray([h["pde_loss"] for h in history], dtype=np.float32),
    lambda_pde=np.asarray([h["lambda_pde"] for h in history], dtype=np.float32),
    valid_mse_phys=np.asarray([h["valid_mse"] for h in history], dtype=np.float32),
)

Epoch 0001 | data loss=7.683e-01 | bc loss=8.257e-01 | PDE loss=1.014e-02 | valid MSE=4.362e+03 | valid rel L2=2.413e-01 | pressure=1.475e-01, u=2.794e-01, v=9.382e-01 | Time: 0:00:21.892565
Epoch 0010 | data loss=1.730e-01 | bc loss=9.548e-02 | PDE loss=3.273e-03 | valid MSE=4.793e+02 | valid rel L2=7.757e-02 | pressure=4.871e-02, u=7.066e-02, v=6.612e-01 | Time: 0:03:39.438857
Epoch 0020 | data loss=1.355e-01 | bc loss=7.429e-02 | PDE loss=2.751e-03 | valid MSE=3.329e+02 | valid rel L2=6.871e-02 | pressure=4.074e-02, u=3.499e-02, v=6.067e-01 | Time: 0:07:19.554641
Epoch 0030 | data loss=1.141e-01 | bc loss=5.752e-02 | PDE loss=3.744e-03 | valid MSE=3.628e+02 | valid rel L2=7.009e-02 | pressure=4.253e-02, u=3.754e-02, v=5.707e-01 | Time: 0:10:59.767517
Epoch 0040 | data loss=9.363e-02 | bc loss=4.967e-02 | PDE loss=3.100e-03 | valid MSE=2.733e+02 | valid rel L2=6.764e-02 | pressure=3.687e-02, u=3.307e-02, v=5.216e-01 | Time: 0:14:40.458507
Epoch 0050 | data loss=7.404e-02 | bc loss=4.

In [ ]:
# Plot training/validation loss trend over iterations (epochs).
epochs = np.asarray([h["epoch"] for h in history])
data_loss_arr = np.asarray([h["data_loss"] for h in history])
bc_loss_arr = np.asarray([h["bc_loss"] for h in history])
pde_loss_arr = np.asarray([h["pde_loss"] for h in history])
train_loss_arr = data_loss_arr + bc_loss_arr + pde_loss_arr
valid_loss_arr = np.asarray([h["valid_mse"] for h in history])
valid_rel_l2_arr = np.asarray([h["valid_relative_l2"] for h in history])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)

axes[0].plot(epochs, data_loss_arr, label="Train data loss", linewidth=2)
axes[0].plot(epochs, bc_loss_arr, label="Train BC loss", linewidth=2)
axes[0].plot(epochs, pde_loss_arr, label="Train PDE loss", linewidth=2)
axes[0].plot(epochs, train_loss_arr, label="Train loss", linewidth=2)
axes[0].plot(epochs, valid_loss_arr, label="Valid MSE", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training/Validation MSE Trend")
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale("log")
axes[0].legend()

axes[1].plot(epochs, valid_rel_l2_arr, label="Valid rel L2", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Relative L2")
axes[1].set_title("Validation Relative L2 Trend")
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale("log")
axes[1].legend()
plt.savefig(output_dir / "loss_history.png")
plt.show()


## Load best checkpoint

In [ ]:
output_dir = Path("results/073126_1")
checkpoint_path = output_dir / "checkpoint.pt"
ckpt = torch.load(checkpoint_path, map_location=DEVICE)

model = DeepONet(**ckpt["model_config"]).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

y_normalizer = FeatureNormalizer.from_state_dict(ckpt["y_normalizer"]).to(DEVICE)
local_aspect_mean = float(ckpt["local_aspect_mean"])
local_aspect_std = float(ckpt["local_aspect_std"])

print("Loaded:", checkpoint_path)

/tmp/ipykernel_12949/2186076967.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location=DEVICE)


Loaded: results/073126_1/checkpoint.pt


## Predict validation case

In [ ]:
output_channel_names = list(valid_data["output_channel_names"])
    
for c in VALID_CASES:
    valid_case_sample_ids = sorted(
        (i for i, mm in enumerate(valid_data["metadata"]) if mm["case_id"] == c),
        key=lambda i: int(valid_data["metadata"][i]["subdomain_id"]),
    )
    metadata = valid_data["metadata"][valid_case_sample_ids[0]]

    plot_data = collect_predictions_for_data(
        model=model,
        data=valid_data,
        device=DEVICE,
        y_normalizer=y_normalizer,
        local_aspect_mean=local_aspect_mean,
        local_aspect_std=local_aspect_std,
        sample_indices=valid_case_sample_ids,
    )

    print("Total plotted points:", plot_data["x"].shape[0])
    print("x range:", plot_data["x"].min(), plot_data["x"].max())
    print("y range:", plot_data["y"].min(), plot_data["y"].max())

    for field_name in output_channel_names:
        field_idx = output_channel_names.index(field_name)

        plot_prediction_imshow_from_points(
            x=plot_data["x"],
            y=plot_data["y"],
            pred=plot_data["pred"],
            truth=plot_data["truth"],
            field_name=field_name,
            field_idx=field_idx,
            n_x_plot=2000,
            n_y_plot=200,
            metadata=plot_data["metadata"],
            metis_cut_face_midpoints=plot_data.get("metis_cut_face_midpoints"),
            interface_placement=plot_data.get("interface_placement"),
            mesh_h5=case_files[metadata["case_id"]]["mesh"],
            coord_unit="mm",
            output_dir=output_dir / "predictions" / "validation",
            filename_prefix=f"valid_case{int(c.split('_')[-1]):02d}",
        )

# Predict test case

In [ ]:
test_data = build_fluent_deeponet_dataset(
    case_files=case_files,
    case_ids=TEST_CASES,
    n_subdomains="adaptive",
    n_interface_points=N_INTERFACE_POINTS,
    n_boundary_points=N_WALL_POINTS,
    interface_placement="fixed",
    interface_jitter=0,
    insert_sharp_control_point_interfaces=False,
    horizontal_interface=HORIZONTAL_INTERFACE,
    field_map=FIELD_MAP,
    bc_kwargs=bc_kwargs,
    keep_raw_case_data=False,
    rng=rng,
)
output_channel_names = test_data["output_channel_names"]
print("Test samples:", len(test_data["samples"]))
print("Test cases:", sorted({int(m["case_id"].split("_")[-1]) for m in test_data["metadata"]}))

Setting n_realizations to 1 for fixed interface placement with no jitter
Processing case channel_18 realization=0 (n_subdomains=10)
Processing case channel_22 realization=0 (n_subdomains=10)
Processing case channel_28 realization=0 (n_subdomains=10)
Processing case channel_31 realization=0 (n_subdomains=10)
Processing case channel_33 realization=0 (n_subdomains=10)
Processing case channel_45 realization=0 (n_subdomains=10)
Processing case channel_51 realization=0 (n_subdomains=10)
Processing case channel_71 realization=0 (n_subdomains=10)
Processing case channel_81 realization=0 (n_subdomains=10)
Processing case channel_89 realization=0 (n_subdomains=10)
Processing case channel_90 realization=0 (n_subdomains=10)
Processing case channel_97 realization=0 (n_subdomains=10)
Processing case channel_111 realization=0 (n_subdomains=10)
Processing case channel_117 realization=0 (n_subdomains=10)
Processing case channel_118 realization=0 (n_subdomains=10)
Processing case channel_133 realization

In [ ]:
l2 = []
for c in TEST_CASES:
    sample_test_ids = sorted(
        (i for i, mm in enumerate(test_data["metadata"]) if mm["case_id"] == c),
        key=lambda i: int(test_data["metadata"][i]["subdomain_id"]),
    )
    if not sample_test_ids:
        continue
    metadata = test_data["metadata"][sample_test_ids[0]]
    

    plot_data = collect_predictions_for_data(
        model=model,
        data=test_data,
        device=DEVICE,
        y_normalizer=y_normalizer,
        local_aspect_mean=local_aspect_mean,
        local_aspect_std=local_aspect_std,
        sample_indices=sample_test_ids,
    )

    print("Total plotted points:", plot_data["x"].shape[0])
    print("x range:", plot_data["x"].min(), plot_data["x"].max())
    print("y range:", plot_data["y"].min(), plot_data["y"].max())
    
    pred = plot_data["pred"]
    truth = plot_data["truth"]
    err = np.abs(pred - truth)
    rel_l2 = np.linalg.norm(err, axis=0) / np.maximum(np.linalg.norm(truth, axis=0), 1.0e-12)
    l2.append(rel_l2)
    overall_avg_rel_l2 = float(np.mean(rel_l2))
    max_field_err = err.max(axis=0)
    print(f"Test case {c} rel l2: {rel_l2}")
    print(f"Test case {c} overall avg rel L2: {overall_avg_rel_l2:.3e}")
    print(f"Test case {c} max field err: {max_field_err}")

    for field_name in output_channel_names:
        field_idx = output_channel_names.index(field_name)

        plot_prediction_imshow_from_points(
            x=plot_data["x"],
            y=plot_data["y"],
            pred=plot_data["pred"],
            truth=plot_data["truth"],
            field_name=field_name,
            field_idx=field_idx,
            metadata=plot_data["metadata"],
            # metis_cut_face_midpoints=plot_data.get("metis_cut_face_midpoints"),
            # interface_placement=plot_data.get("interface_placement"),
            n_x_plot=2000,
            n_y_plot=200,
            coord_unit="mm",
            output_dir=output_dir / "predictions" / "test_ablation",
            filename_prefix=f"test_case{int(c.split('_')[-1]):02d}",
            mesh_h5=case_files[test_data["metadata"][sample_test_ids[0]]["case_id"]]["mesh"],
        )

l2 = np.array(l2)
l2_err = np.mean(l2, axis=0)
l2_err_max = np.max(l2, axis=0)
l2_err_min = np.min(l2, axis=0)

print(l2_err)
print(l2_err_max)

Total plotted points: 240746
x range: 0.00026093746 0.99973804
y range: -0.0032173737 0.103217304
Test case channel_18 rel l2: [0.00726027 0.00460894 0.10445481]
Test case channel_18 overall avg rel L2: 3.877e-02
Test case channel_18 max field err: [1.2348416e+03 2.5969279e-01 1.6414016e-01]
Total plotted points: 226940
x range: 0.00024718884 0.9997411
y range: -0.0022469538 0.10224443
Test case channel_22 rel l2: [0.00790659 0.00352925 0.10028195]
Test case channel_22 overall avg rel L2: 3.724e-02
Test case channel_22 max field err: [1.6189702e+03 2.8494000e-01 1.9204223e-01]
Total plotted points: 231186
x range: 0.0002531517 0.9997418
y range: -0.0042465804 0.104217246
Test case channel_28 rel l2: [0.01749061 0.00879564 0.20060077]
Test case channel_28 overall avg rel L2: 7.563e-02
Test case channel_28 max field err: [2.6971860e+03 2.7112323e-01 2.6677406e-01]
Total plotted points: 228072
x range: 0.00025701115 0.9997439
y range: -0.0015170992 0.1015051
Test case channel_31 rel l2: [

# Test Ablation

In [ ]:
ROOT_DIR = Path("/home/hantianl/Documents/PIDIF/")
DATASET_NAME = "channel_water_ablation"

def case_paths(ch):
    return {
        "design": ROOT_DIR / f"2d_geometry_specs/{DATASET_NAME}/{ch}.json",
        "mesh": ROOT_DIR / f"runs_2d/{DATASET_NAME}/{ch}/{ch}.msh.h5",
        "dat": ROOT_DIR / f"runs_2d/{DATASET_NAME}/{ch}/case2d.dat.h5",
    }

# Keep only cases whose design/mesh/dat files all exist (completed runs).
available_cases = [
    ch.name for ch in (ROOT_DIR / "runs_2d" / DATASET_NAME).iterdir()
    if all(case_paths(ch.name)[k].exists() for k in ("design", "mesh", "dat"))
]
case_files = {ch: case_paths(ch) for ch in available_cases}

print(f"Available cases ({len(available_cases)}):")
print(sorted([int(ch.split("_")[-1]) for ch in available_cases]))

Available cases (50):
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]


In [ ]:
TEST_CASES = [f"channel_{i:03d}" for i in range(50)]  # Test on channels 1-10

test_data = build_fluent_deeponet_dataset(
    case_files=case_files,
    case_ids=TEST_CASES,
    n_subdomains="adaptive",
    n_interface_points=N_INTERFACE_POINTS,
    n_boundary_points=N_WALL_POINTS,
    interface_placement="fixed",
    interface_jitter=0,
    insert_sharp_control_point_interfaces=False,
    horizontal_interface=HORIZONTAL_INTERFACE,
    field_map=FIELD_MAP,
    bc_kwargs=bc_kwargs,
    keep_raw_case_data=False,
    rng=rng,
)
output_channel_names = test_data["output_channel_names"]
print("Test samples:", len(test_data["samples"]))
print("Test cases:", sorted({int(m["case_id"].split("_")[-1]) for m in test_data["metadata"]}))

Setting n_realizations to 1 for fixed interface placement with no jitter
Processing case channel_000 realization=0 (n_subdomains=5)
Processing case channel_001 realization=0 (n_subdomains=5)
Processing case channel_002 realization=0 (n_subdomains=5)
Processing case channel_003 realization=0 (n_subdomains=5)
Processing case channel_004 realization=0 (n_subdomains=5)
Processing case channel_005 realization=0 (n_subdomains=5)
Processing case channel_006 realization=0 (n_subdomains=5)
Processing case channel_007 realization=0 (n_subdomains=5)
Processing case channel_008 realization=0 (n_subdomains=5)
Processing case channel_009 realization=0 (n_subdomains=5)
Processing case channel_010 realization=0 (n_subdomains=20)
Processing case channel_011 realization=0 (n_subdomains=20)
Processing case channel_012 realization=0 (n_subdomains=20)
Processing case channel_013 realization=0 (n_subdomains=20)
Processing case channel_014 realization=0 (n_subdomains=20)
Processing case channel_015 realizati

In [ ]:
l2 = []
for c in TEST_CASES:
    sample_test_ids = sorted(
        (i for i, mm in enumerate(test_data["metadata"]) if mm["case_id"] == c),
        key=lambda i: int(test_data["metadata"][i]["subdomain_id"]),
    )
    if not sample_test_ids:
        continue
    metadata = test_data["metadata"][sample_test_ids[0]]
    

    plot_data = collect_predictions_for_data(
        model=model,
        data=test_data,
        device=DEVICE,
        y_normalizer=y_normalizer,
        local_aspect_mean=local_aspect_mean,
        local_aspect_std=local_aspect_std,
        sample_indices=sample_test_ids,
    )

    print("Total plotted points:", plot_data["x"].shape[0])
    print("x range:", plot_data["x"].min(), plot_data["x"].max())
    print("y range:", plot_data["y"].min(), plot_data["y"].max())
    
    pred = plot_data["pred"]
    truth = plot_data["truth"]
    err = np.abs(pred - truth)
    rel_l2 = np.linalg.norm(err, axis=0) / np.maximum(np.linalg.norm(truth, axis=0), 1.0e-12)
    l2.append(rel_l2)
    overall_avg_rel_l2 = float(np.mean(rel_l2))
    max_field_err = err.max(axis=0)
    print(f"Test case {c} rel l2: {rel_l2}")
    print(f"Test case {c} overall avg rel L2: {overall_avg_rel_l2:.3e}")
    print(f"Test case {c} max field err: {max_field_err}")

    for field_name in output_channel_names:
        field_idx = output_channel_names.index(field_name)

        plot_prediction_imshow_from_points(
            x=plot_data["x"],
            y=plot_data["y"],
            pred=plot_data["pred"],
            truth=plot_data["truth"],
            field_name=field_name,
            field_idx=field_idx,
            metadata=plot_data["metadata"],
            # metis_cut_face_midpoints=plot_data.get("metis_cut_face_midpoints"),
            # interface_placement=plot_data.get("interface_placement"),
            n_x_plot=2000,
            n_y_plot=200,
            coord_unit="mm",
            output_dir=output_dir / "predictions" / "test_ablation",
            filename_prefix=f"test_case{int(c.split('_')[-1]):02d}",
            mesh_h5=case_files[test_data["metadata"][sample_test_ids[0]]["case_id"]]["mesh"],
        )

l2 = np.array(l2)
l2_err = np.mean(l2, axis=0)
l2_err_max = np.max(l2, axis=0)
l2_err_min = np.min(l2, axis=0)

print(l2_err)
print(l2_err_max)